# AI動画生成 無料GPUノートブック（Kaggle / Colab）

リアル系の短尺動画（犬など）を**クラウドの無料GPU**で生成する。
MacBook（Apple Silicon）ローカルでは実用速度が出ないための**クラウドルート**。

## 事前設定（重要）
- **Kaggle**: 右パネル Session options → Accelerator: **GPU T4 x2** か **P100** / Internet: **ON**（無料枠: 週30時間）
- **Colab**: ランタイム → ランタイムのタイプを変更 → **T4 GPU**（無料枠は変動・切断されやすい）

## T4を使ううえでの制約（設計上の前提）
T4はTuring世代なので **bf16もFP8も非対応**。ネット上の作例の多くはbf16/FP8前提なので、
そのままコピペすると落ちる。このノートブックは自動でfp16に切り替える。
またWan 2.2はVAEデコード時にVRAMを使い切るOOM報告があるため、タイリングを有効化している。

## 使い方
上から順にセルを実行 → プロンプトを書く → 生成 → zipでダウンロード。
初回はモデル重み（十数GB）のDLで10〜20分かかる。**プロンプトをまとめて回すのが無料枠の節約になる。**

> このノートブックはGPUの無い環境で作成したため **実行未検証**。
> エラーが出た場合はセルの出力メッセージを添えて相談してください。


In [ ]:
# 1) GPU環境の確認とdtypeの自動選択
import torch, os

if not torch.cuda.is_available():
    raise SystemExit('GPUが見えていません。アクセラレータ設定(T4/P100)を確認してセッションを再起動してください')

props   = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / 1e9
BF16    = torch.cuda.is_bf16_supported()
DTYPE   = torch.bfloat16 if BF16 else torch.float16   # T4はbf16非対応なのでfp16に落ちる

WORK = '/content' if os.path.exists('/content') else '/kaggle/working'
OUT_DIR = os.path.join(WORK, 'video_out')
os.makedirs(OUT_DIR, exist_ok=True)

print(f'GPU   : {props.name}')
print(f'VRAM  : {VRAM_GB:.1f} GB')
print(f'bf16  : {BF16}  -> dtype={str(DTYPE).replace("torch.","")}')
print(f'出力先: {OUT_DIR}')


In [ ]:
# 2) 依存のインストール（2〜3分）
# diffusersはWan 2.2対応版が必要。PyPI版で動かない場合は下の git+ 行に切り替える。
!pip install -q 'diffusers>=0.35' 'transformers>=4.49' accelerate safetensors sentencepiece ftfy imageio imageio-ffmpeg
# !pip install -q git+https://github.com/huggingface/diffusers.git  # ← PyPI版でクラス未定義エラーが出た場合

import diffusers, transformers
print('diffusers   :', diffusers.__version__)
print('transformers:', transformers.__version__)


In [ ]:
# 3) 生成設定（ここだけ触ればよい）

MODEL = 'wan'        # 'wan' = 高品質・低速 / 'ltx' = 低品質・高速（試し撮り向き）

# 縦型（Reels / Shorts）。Wanは16の倍数、LTXは32の倍数である必要がある
WIDTH, HEIGHT = 480, 832

# Wanは 4n+1、LTXは 8n+1 のフレーム数にすること
NUM_FRAMES = 81 if MODEL == 'wan' else 121
FPS        = 24

# distill版LTXは guidance=1.0 / steps=4〜10 が指定。Wanは通常のCFG
STEPS    = 30  if MODEL == 'wan' else 8
GUIDANCE = 5.0 if MODEL == 'wan' else 1.0

SEED = 12345          # 同じ値なら同じ絵が出る。変えると別パターン
USE_IMAGE = False     # True にすると参照画像から動かす（i2v）

NEGATIVE = (
    '低画質, ぼやけ, 崩れた顔, 余分な脚, 指の破綻, 文字, ロゴ, 透かし, '
    'blurry, low quality, deformed, extra limbs, distorted anatomy, watermark, text'
)

print(f'{MODEL} / {WIDTH}x{HEIGHT} / {NUM_FRAMES}f @{FPS}fps = {NUM_FRAMES/FPS:.1f}秒 / steps={STEPS}')


In [ ]:
# 4) パイプラインの読み込み（初回はモデルDLで10〜20分）
import torch
from diffusers.utils import export_to_video

if MODEL == 'wan':
    from diffusers import AutoencoderKLWan, WanPipeline, WanImageToVideoPipeline
    MODEL_ID = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
    # VAEはfp32で読むのが公式作例の指定（fp16だと出力が破綻しやすい）
    vae  = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
    Cls  = WanImageToVideoPipeline if USE_IMAGE else WanPipeline
    pipe = Cls.from_pretrained(MODEL_ID, vae=vae, torch_dtype=DTYPE)
else:
    from diffusers import LTXPipeline, LTXImageToVideoPipeline
    MODEL_ID = 'Lightricks/LTX-Video-0.9.7-distilled'
    Cls  = LTXImageToVideoPipeline if USE_IMAGE else LTXPipeline
    pipe = Cls.from_pretrained(MODEL_ID, torch_dtype=DTYPE)

# 16GBに収めるための省メモリ設定
pipe.enable_model_cpu_offload()        # 使わない部分はCPU RAMへ退避
try:
    pipe.vae.enable_tiling()           # VAEデコード時のOOM対策（diffusers #12097）
except Exception as e:
    print('tiling有効化をスキップ:', e)

print('読み込み完了:', MODEL_ID)


In [ ]:
# 5) 【任意】参照画像のアップロード（USE_IMAGE = True のときだけ）
#    静止画から動かすと、狙った見た目を保ちやすい
import os, glob
from PIL import Image

REF_DIR = os.path.join(WORK, 'video_input')
os.makedirs(REF_DIR, exist_ok=True)

ref_image = None
if USE_IMAGE:
    files = sorted(glob.glob(os.path.join(REF_DIR, '*.[jp][pn]g')))
    if not files:
        raise SystemExit(f'{REF_DIR} に画像を置いてください（Kaggleは右パネルのUploadから）')
    ref_image = Image.open(files[0]).convert('RGB').resize((WIDTH, HEIGHT))
    print('参照画像:', files[0])
else:
    print('USE_IMAGE = False のためスキップ（テキストのみで生成）')


In [ ]:
# 6) プロンプトを並べて一括生成
#    無料枠の節約のため、1セッションでまとめて回すこと
import os, time, torch
from diffusers.utils import export_to_video

PROMPTS = [
    'A fluffy shiba inu puppy sitting on a wooden floor, tilting its head curiously, '
    'soft natural window light, shallow depth of field, photorealistic, 4k',

    'A golden retriever running toward the camera across a green lawn in slow motion, '
    'sunny afternoon, fur moving naturally, photorealistic handheld shot',
]

for i, prompt in enumerate(PROMPTS, 1):
    t0 = time.time()
    gen = torch.Generator(device='cpu').manual_seed(SEED + i)
    kwargs = dict(
        prompt=prompt, negative_prompt=NEGATIVE,
        width=WIDTH, height=HEIGHT, num_frames=NUM_FRAMES,
        num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=gen,
    )
    if USE_IMAGE and ref_image is not None:
        kwargs['image'] = ref_image

    print(f'[{i}/{len(PROMPTS)}] 生成中... {prompt[:50]}...')
    frames = pipe(**kwargs).frames[0]

    path = os.path.join(OUT_DIR, f'clip_{i:03d}.mp4')
    export_to_video(frames, path, fps=FPS)
    print(f'  -> {path}  ({time.time()-t0:.0f}秒)')

    torch.cuda.empty_cache()

print('完了')


In [ ]:
# 7) 出力をzipにまとめてダウンロード
import glob, shutil, os

clips = sorted(glob.glob(os.path.join(OUT_DIR, '*.mp4')))
print(f'{len(clips)}本のクリップ:')
for c in clips:
    print(' ', os.path.basename(c), f'{os.path.getsize(c)/1e6:.1f} MB')

zip_base = os.path.join(WORK, 'video_clips')
shutil.make_archive(zip_base, 'zip', OUT_DIR)
print(f'\nzip: {zip_base}.zip  ({os.path.getsize(zip_base + ".zip")/1e6:.1f} MB)')
print('Kaggle: 右パネル Output からダウンロード / Colab: 左のファイルペインから')


## 次の手順（MacBook側・GPU不要）

zipを展開して、リポジトリの仕上げスクリプトにかける:

```bash
# 1) 初回のみ: Mac環境のセットアップ
bash scripts/setup_mac_video.sh

# 2) クリップをReels/Shorts形式(1080x1920)に整える
python scripts/finish_reel.py video_input/clip_001.mp4 -o video_out/reel_001.mp4

# 3) 複数クリップをつなぐ場合
python scripts/finish_reel.py video_input/*.mp4 --concat -o video_out/reel_final.mp4
```

詳細は `docs/06_ai_video_pipeline.md` を参照。
